# DC4 Input Comparison Notebook (FC, FPC, NDVI)

This notebook is designed to **compare DC4 composite inputs and outputs** across the three time-series modes used in the EDS/SLATS pipeline: **FC**, **FPC**, and **NDVI**. Its purpose is to help diagnose inconsistencies in date coverage, input selection, and metadata handling between modes—particularly when NDVI appears to produce fewer DC4 images than FC/FPC.

## What this notebook checks

### 1. DC4 File Discovery
Identifies all `dc4*.img` products generated for each mode (`fc`, `fpc`, `ndvi`) within a given scene (e.g. `p089r084`) and reports how many were created.

### 2. Date Coverage Comparison
Extracts acquisition dates from DC4 filenames and compares:
- Total number of dates per mode  
- Overlapping dates between modes  
- Dates present in FC/FPC but missing in NDVI (and vice versa)

This quickly reveals whether all modes are drawing from the same temporal baseline.

### 3. Presence Matrix
Builds a simple **date × mode** table showing which DC4 dates exist for each mode, making gaps or asymmetries visually obvious.

### 4. Manifest (eds_master_results) Inspection
Reads the most recent `eds_master_results_*.json` for each mode and summarises:
- Requested vs effective start/end dates  
- Seasonal window and lookback used  
- Inputs recorded by the pipeline (e.g. SR items used for NDVI, FC/FPC match counts and patterns)

This shows what each run *claims* it used as inputs.

### 5. Raster Metadata Sanity Check
For a small sample of DC4 rasters per mode, inspects:
- Raster dimensions and band count  
- NoData values  
- Projection and geotransform  

This helps confirm that outputs are spatially consistent across modes.

### 6. Optional Pixel Sampling
Randomly samples a small number of pixels from representative DC4 rasters in each mode to compare basic statistics (min, max, mean, median). This is a lightweight sanity check to detect gross differences.

## When to use this
Use this notebook when:
- NDVI produces far fewer DC4 images than FC/FPC  
- You suspect different seasonal filtering or baseline selection between modes  
- You want to confirm whether FC, FPC, and NDVI are truly using comparable inputs  

The results provide concrete evidence for whether discrepancies originate in **input discovery**, **seasonal window logic**, or **naming/metadata differences**.


In [1]:
from pathlib import Path

# ---- EDIT THESE ----
SCENE = "p089r084"

# This is the OUT root you pass to the master pipeline (the folder that contains fc/, fpc/, ndvi/)
OUT_ROOT = Path("/home/jovyan/work-easi-eds/data/compat/files")

# Optional: if you want to restrict to a particular tile/date run, set these (or leave None)
START_DATE = None  # e.g. "20230125"
END_DATE   = None  # e.g. "20231024"

MODES = ["fc", "fpc", "ndvi"]  # expected folders under OUT_ROOT



In [2]:
import re
import glob
from datetime import datetime

DATE_RX = re.compile(r"(19|20)\d{6}")  # YYYYMMDD

def extract_date_from_name(name: str) -> str | None:
    m = DATE_RX.search(name)
    return m.group(0) if m else None

def find_dc4_files(mode_dir: Path):
    """
    Find dc4*.img files in a mode/scene directory.
    Handles dc4fc, dc4fpc, dc4ndvi, dc4mz variants.
    """
    scene_dir = mode_dir / SCENE
    pats = [
        str(scene_dir / f"lztmre_{SCENE}_*_dc4*.img"),
        str(scene_dir / f"*{SCENE}*_dc4*.img"),
    ]
    files = []
    for p in pats:
        files.extend(glob.glob(p))
    # Deduplicate + sort
    files = sorted({str(Path(f)) for f in files})
    return [Path(f) for f in files]

def mode_scene_dir(mode: str) -> Path:
    return OUT_ROOT / mode / SCENE

def find_manifests(mode: str):
    """
    Find eds_master_results_*.json in OUT_ROOT/mode/SCENE.
    If START_DATE/END_DATE set, try to prefer a matching file.
    """
    d = OUT_ROOT / mode / SCENE
    cands = sorted(d.glob("eds_master_results_*.json"))
    if not cands:
        return []

    if START_DATE and END_DATE:
        key = f"d{START_DATE}_{END_DATE}"
        preferred = [p for p in cands if key in p.name]
        if preferred:
            return preferred
    return cands


In [3]:
import pandas as pd

rows = []
dc4_by_mode = {}

for mode in MODES:
    dc4s = find_dc4_files(OUT_ROOT / mode)
    dc4_by_mode[mode] = dc4s

    for fp in dc4s:
        d = extract_date_from_name(fp.name)
        tag = fp.name.split("_")[-1].replace(".img", "")  # e.g. dc4fpc
        rows.append({
            "mode": mode,
            "file": str(fp),
            "date": d,
            "tag": tag,
        })

df = pd.DataFrame(rows)

# Basic view
print("dc4 file counts by mode:")
display(df.groupby("mode")["file"].count().rename("dc4_count").to_frame())

print("\nSample rows:")
display(df.sort_values(["mode", "date"]).head(10))


dc4 file counts by mode:


,dc4_count
mode,
fc,145
fpc,145
ndvi,105



Sample rows:


,mode,file,date,tag
0,fc,/home/jovyan/work-easi-eds/data/compat/files/f...,20151213,dc4fc
1,fc,/home/jovyan/work-easi-eds/data/compat/files/f...,20151229,dc4fc
2,fc,/home/jovyan/work-easi-eds/data/compat/files/f...,20160130,dc4fc
3,fc,/home/jovyan/work-easi-eds/data/compat/files/f...,20160215,dc4fc
4,fc,/home/jovyan/work-easi-eds/data/compat/files/f...,20160302,dc4fc
5,fc,/home/jovyan/work-easi-eds/data/compat/files/f...,20160318,dc4fc
6,fc,/home/jovyan/work-easi-eds/data/compat/files/f...,20160419,dc4fc
7,fc,/home/jovyan/work-easi-eds/data/compat/files/f...,20160505,dc4fc
8,fc,/home/jovyan/work-easi-eds/data/compat/files/f...,20160521,dc4fc
9,fc,/home/jovyan/work-easi-eds/data/compat/files/f...,20160724,dc4fc


In [4]:
def dates_for(mode: str) -> set[str]:
    sub = df[df["mode"] == mode]
    return set(x for x in sub["date"].dropna().tolist())

date_sets = {m: dates_for(m) for m in MODES}

print("Unique date counts:")
for m in MODES:
    print(f"  {m}: {len(date_sets[m])}")

# Pairwise overlaps
print("\nPairwise overlap counts:")
for i, a in enumerate(MODES):
    for b in MODES[i+1:]:
        inter = date_sets[a] & date_sets[b]
        print(f"  {a} ∩ {b}: {len(inter)}")

# What’s missing where
print("\nMissing dates (A - B):")
for i, a in enumerate(MODES):
    for b in MODES:
        if a == b:
            continue
        miss = sorted(date_sets[a] - date_sets[b])
        print(f"  {a} missing in {b}: {len(miss)}")
        if miss[:15]:
            print("   ", ", ".join(miss[:15]), ("..." if len(miss) > 15 else ""))


Unique date counts:
  fc: 145
  fpc: 145
  ndvi: 105

Pairwise overlap counts:
  fc ∩ fpc: 145
  fc ∩ ndvi: 105
  fpc ∩ ndvi: 105

Missing dates (A - B):
  fc missing in fpc: 0
  fc missing in ndvi: 40
    20231117, 20231125, 20231211, 20231227, 20240112, 20240120, 20240128, 20240221, 20240308, 20240324, 20240417, 20240425, 20240527, 20240706, 20240714 ...
  fpc missing in fc: 0
  fpc missing in ndvi: 40
    20231117, 20231125, 20231211, 20231227, 20240112, 20240120, 20240128, 20240221, 20240308, 20240324, 20240417, 20240425, 20240527, 20240706, 20240714 ...
  ndvi missing in fc: 0
  ndvi missing in fpc: 0


In [5]:
all_dates = sorted(set().union(*date_sets.values()))
mat = pd.DataFrame({"date": all_dates})

for m in MODES:
    mat[m] = mat["date"].isin(date_sets[m])

print("Presence matrix (first 50 dates):")
display(mat.head(50))


Presence matrix (first 50 dates):


,date,fc,fpc,ndvi
0,20151213,True,True,True
1,20151229,True,True,True
2,20160130,True,True,True
3,20160215,True,True,True
4,20160302,True,True,True
5,20160318,True,True,True
6,20160419,True,True,True
7,20160505,True,True,True
8,20160521,True,True,True
9,20160724,True,True,True


In [6]:
import json

def load_json(p: Path):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

for mode in MODES:
    mans = find_manifests(mode)
    if not mans:
        print(f"\n[{mode}] No manifests found in: {OUT_ROOT / mode / SCENE}")
        continue

    # Use the newest by mtime if more than one
    mans = sorted(mans, key=lambda p: p.stat().st_mtime, reverse=True)
    mpath = mans[0]
    data = load_json(mpath)

    print("\n" + "="*100)
    print(f"[{mode}] manifest: {mpath}")
    print(f" tile: {data.get('tile')}, timeseries_source: {data.get('timeseries_source')}")
    print(f" requested: {data.get('requested_start_date')} -> {data.get('requested_end_date')}")
    print(f" effective: {data.get('effective_start_date')} -> {data.get('effective_end_date')}")
    print(f" window: {data.get('seasonal_window', {}).get('window_start_mmdd')} -> {data.get('seasonal_window', {}).get('window_end_mmdd')}")
    print(f" lookback_used: {data.get('seasonal_window', {}).get('lookback_used')}")

    # SR start/end
    sr_inputs = data.get("sr_inputs", {})
    if sr_inputs:
        print("\n SR inputs:")
        print("  start:", sr_inputs.get("start_path"))
        print("  end:  ", sr_inputs.get("end_path"))

    # If your pipeline wrote compat_build block:
    cb = (data.get("inputs", {}) or {}).get("compat_build", {})
    if cb:
        print("\n compat_build:")
        for k in ["mode", "dc4_tag", "sr_only_clr", "baseline_start_date", "baseline_end_date"]:
            if k in cb:
                print(f"  {k}: {cb[k]}")

        # NDVI run usually has sr_items_used
        if "sr_items_used" in cb:
            print(f"  sr_items_used: {len(cb['sr_items_used'])}")
            for item in cb["sr_items_used"][:10]:
                print("   ", item["date"], item["path"])
            if len(cb["sr_items_used"]) > 10:
                print("    ...")

        # FC/FPC run may have patterns and matched_files under inputs[fc]/inputs[fpc]
    inputs = data.get("inputs", {})
    if "fc" in inputs:
        fc = inputs["fc"]
        if isinstance(fc, dict):
            print("\n inputs['fc'] summary:")
            if "matched_count" in fc:
                print("  matched_count:", fc["matched_count"])
            if "patterns" in fc:
                print("  patterns (first 5):")
                for p in fc["patterns"][:5]:
                    print("   ", p)

    if "fpc" in inputs:
        fpc = inputs["fpc"]
        if isinstance(fpc, dict):
            print("\n inputs['fpc'] summary:")
            if "matched_count" in fpc:
                print("  matched_count:", fpc["matched_count"])
            if "patterns" in fpc:
                print("  patterns (first 5):")
                for p in fpc["patterns"][:5]:
                    print("   ", p)



[fc] manifest: /home/jovyan/work-easi-eds/data/compat/files/fc/p089r084/eds_master_results_089_084_fc_d20230125_20231024.json
 tile: 089_084, timeseries_source: fc
 requested: 20230125 -> 20231024
 effective: 20230125 -> 20231024
 window: 1125 -> 1224
 lookback_used: 10

 SR inputs:
  start: /home/jovyan/scratch/eds/tiles/p089r084/sr/2023/202301/ls89sr_p089r084_20230125_nbart6m6_clr.tif
  end:   /home/jovyan/scratch/eds/tiles/p089r084/sr/2023/202310/ls89sr_p089r084_20231024_nbart6m6_clr.tif

 compat_build:
  mode: fc
  dc4_tag: dc4fc

 inputs['fc'] summary:
  matched_count: 145
  patterns (first 5):
    /home/jovyan/scratch/eds/tiles/p089r084/fc/**/*galsfc3_*_fcm*_clr.tif
    /home/jovyan/scratch/eds/tiles/089_084/fc/**/*galsfc3_*_fcm*_clr.tif
    /home/jovyan/scratch/eds/tiles/089/084/fc/**/*galsfc3_*_fcm*_clr.tif

[fpc] manifest: /home/jovyan/work-easi-eds/data/compat/files/fpc/p089r084/eds_master_results_089_084_fpc_d20230125_20231024.json
 tile: 089_084, timeseries_source: fpc
 re